In [221]:
from pathlib import Path
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
import optuna

from sklearn.preprocessing import PolynomialFeatures
from feature_engine.encoding import RareLabelEncoder, CountFrequencyEncoder
from feature_engine.discretisation import EqualFrequencyDiscretiser
from feature_engine.selection import DropConstantFeatures

In [193]:
import warnings
warnings.filterwarnings("ignore")

In [194]:
DATA_DIR = Path("../data")
SEED = 42

In [195]:
def get_importances(model, X_train, X_test, y_test, model_name, has_builtin=True):
    """Compute permutation importance, plus gain-based importance if available."""
    # Permutation importance (works for any fitted model)
    perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
    df = pd.DataFrame({
        "feature": X_test.columns,
        f"{model_name}_perm": perm.importances_mean,
    })
    df[f"{model_name}_rank_perm"] = df[f"{model_name}_perm"].rank(ascending=False)

    # Gain-based importance (tree models only)
    if has_builtin:
        df[f"{model_name}_gain"] = model.feature_importances_
        df[f"{model_name}_rank_gain"] = df[f"{model_name}_gain"].rank(ascending=False)

    return df

# Feature Creation Pipelines

In [196]:
def impute_data(data: pd.DataFrame):
    d = data.copy()
    cats = d.select_dtypes(include=["str", "object"]).columns

    for col in d.columns:
        if col in cats:
            d[col] = d[col].fillna("unknown")
            d[col] = d[col].astype("category")
        else:
            d[col] = d[col].fillna(d[col].median())

    return d

data_imputer = FunctionTransformer(impute_data)

In [197]:
def add_specific_features(data: pd.DataFrame):
    d = data.copy()

    d["is_overworked"] = (d["passthrough__hours-per-week"] > 40).astype(int)
    d["net_capital"] = (d["passthrough__capital-gain"] - d["capital_loss"])

    return d

feature_adder = FunctionTransformer(add_specific_features)

In [198]:
cat_selector = make_column_selector(dtype_include=["object", "category"])
num_selector = make_column_selector(
    pattern=r"^(?!.*(?:gender|capital-gain|capital-loss)).*$",
    dtype_include="number",
)

cat_pipeline = Pipeline([
    ("rare_cat", RareLabelEncoder(tol=0.01)),
    ("freq_cat", CountFrequencyEncoder(encoding_method="frequency")),
])

num_pipeline = Pipeline([
    ("discretize", EqualFrequencyDiscretiser(q=5)),
])

ct = ColumnTransformer(
        transformers=[
            ("cat", cat_pipeline, cat_selector),
            ("num", num_pipeline, num_selector),
        ],
        remainder="passthrough",
    ).set_output(transform="pandas")

preprocessor = Pipeline([
    ("impute", data_imputer),
    ("encode_and_discretize", ct),
    ("drop_consts", DropConstantFeatures()),
])

# Get Training Data

In [199]:
df = pd.read_csv(DATA_DIR / "adult.csv")
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0


In [200]:
X = df.drop(columns=["income", "fnlwgt"])
y = df["income"]

# get splits
X_train_fe, X_test_fe, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# apply preprocessing
X_train_fe, X_test_fe = (preprocessor.fit_transform(X_train_fe), preprocessor.fit_transform(X_test_fe))

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

# num features
len(X_train_fe.columns)

13

In [201]:
X_train_poly_input = X_train_fe.copy()
X_test_poly_input = X_test_fe.copy()

poly = PolynomialFeatures(
    degree=3,
    interaction_only=True,
    include_bias=False
)

# fit on training data only
X_train_poly = poly.fit_transform(X_train_poly_input)

# transform test data using the same fitted transformer
X_test_poly = poly.transform(X_test_poly_input)

# feature names
poly_feature_names = poly.get_feature_names_out(X_train_poly_input.columns)

# convert back to DataFrames
X_train_poly = pd.DataFrame(
    X_train_poly,
    columns=poly_feature_names,
    index=X_train_poly_input.index
)

X_test_poly = pd.DataFrame(
    X_test_poly,
    columns=poly_feature_names,
    index=X_test_poly_input.index
)

print("Selected FE train shape:", X_train_poly_input.shape)
print("Expanded train shape:", X_train_poly.shape)
print("Selected FE test shape:", X_test_poly_input.shape)
print("Expanded test shape:", X_test_poly.shape)

Selected FE train shape: (39073, 13)
Expanded train shape: (39073, 377)
Selected FE test shape: (9769, 13)
Expanded test shape: (9769, 377)


# Benchmarking Engineered Features

In [202]:
fe_xgb = XGBClassifier(
    random_state=SEED,
    enable_categorical=True,
    device="cuda:0",
    tree_method="hist"
)

fe_xgb_cvs = cross_val_score(fe_xgb, X_train_fe, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {fe_xgb_cvs.mean():.4f} +/- {fe_xgb_cvs.std():.4f}")

fe_xgb.fit(X_train_fe, y_train)
fe_xgb_preds = fe_xgb.predict(X_test_fe)

fe_xgb_metric_score = balanced_accuracy_score(y_test, fe_xgb_preds)
print(f"Balanced Accuracy: {fe_xgb_metric_score:.4f}")

cv scores: 0.7973 +/- 0.0052
Balanced Accuracy: 0.7217


In [203]:
nn = MLPClassifier(
    random_state=SEED,
    hidden_layer_sizes=(32, 16),
    learning_rate="adaptive",
    solver="adam",
)

fe_nn = Pipeline(
    [
        ("scale", ColumnTransformer(transformers=[
           ("scale", StandardScaler(), make_column_selector(dtype_include=["number"]))
        ])),
        ("model", nn)
    ]
)

fe_nn_cvs = cross_val_score(fe_nn, X_train_fe, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {fe_nn_cvs.mean():.4f} +/- {fe_nn_cvs.std():.4f}")

fe_nn.fit(X_train_fe, y_train)
fe_nn_preds = fe_nn.predict(X_test_fe)

fe_nn_metric_score = balanced_accuracy_score(y_test, fe_nn_preds)
print(f"Balanced Accuracy: {fe_nn_metric_score:.4f}")

cv scores: 0.7552 +/- 0.0140
Balanced Accuracy: 0.7654


In [225]:
fe_xgb_imp = get_importances(fe_xgb, X_train_fe, X_test_fe, y_test, "xgb", has_builtin=True)
fe_nn_imp  = get_importances(fe_nn, X_train_fe, X_test_fe, y_test, "nn", has_builtin=False)

combined_fe = fe_xgb_imp.merge(fe_nn_imp, on="feature").sort_values("xgb_rank_perm")
combined_fe

,feature,xgb_perm,xgb_rank_perm,xgb_gain,xgb_rank_gain,nn_perm,nn_rank_perm
11,remainder__capital-gain,0.061757,1.0,0.121074,3.0,0.042901,3.0
2,cat__marital-status,0.028754,2.0,0.437702,1.0,0.053547,2.0
8,num__educational-num,0.016993,3.0,0.182244,2.0,0.055430,1.0
12,remainder__capital-loss,0.015467,4.0,0.058812,4.0,0.004484,10.0
7,num__age,0.010810,5.0,0.041730,5.0,0.017392,4.0
9,num__hours-per-week,0.007022,6.0,0.031911,7.0,0.010973,5.0
3,cat__occupation,0.006828,7.0,0.034089,6.0,0.010329,7.0
4,cat__relationship,0.004545,8.0,0.016663,10.0,0.010789,6.0
0,cat__workclass,0.003378,9.0,0.014813,11.0,0.002805,11.0
10,remainder__gender,0.002539,10.0,0.018096,9.0,0.008507,8.0


# Benchmarking Poly Features

In [205]:
poly_xgb = XGBClassifier(
    random_state=SEED,
    enable_categorical=True,
    device="cuda:0",
    tree_method="hist"
)

poly_xgb_cvs = cross_val_score(poly_xgb, X_train_poly, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {poly_xgb_cvs.mean():.4f} +/- {poly_xgb_cvs.std():.4f}")

poly_xgb.fit(X_train_poly, y_train)
poly_xgb_preds = poly_xgb.predict(X_test_poly)

poly_xgb_metric_score = balanced_accuracy_score(y_test, poly_xgb_preds)
print(f"Balanced Accuracy: {poly_xgb_metric_score:.4f}")

cv scores: 0.7892 +/- 0.0053
Balanced Accuracy: 0.7565


In [206]:
nn = MLPClassifier(
    random_state=SEED,
    hidden_layer_sizes=(32, 16),
    learning_rate="adaptive",
    solver="adam",
)

poly_nn = Pipeline(
    [
        ("scale", ColumnTransformer(transformers=[
           ("scale", StandardScaler(), make_column_selector(dtype_include=["number"]))
        ])),
        ("model", nn)
    ]
)

poly_nn_cvs = cross_val_score(poly_nn, X_train_poly, y_train, cv=skf, n_jobs=-1, scoring="balanced_accuracy")
print(f"cv scores: {poly_nn_cvs.mean():.4f} +/- {poly_nn_cvs.std():.4f}")

poly_nn.fit(X_train_poly, y_train)
poly_nn_preds = poly_nn.predict(X_test_poly)

poly_nn_metric_score = balanced_accuracy_score(y_test, poly_nn_preds)
print(f"Balanced Accuracy: {poly_nn_metric_score:.4f}")

cv scores: 0.7585 +/- 0.0129
Balanced Accuracy: 0.7659


In [224]:
poly_xgb_imp = get_importances(poly_xgb, X_train_poly, X_test_poly, y_test, "xgb", has_builtin=True)
poly_nn_imp  = get_importances(poly_nn,  X_train_poly, X_test_poly, y_test, "nn",  has_builtin=False)

combined_poly = poly_xgb_imp.merge(poly_nn_imp, on="feature").sort_values("xgb_rank_perm")
combined_poly

,feature,xgb_perm,xgb_rank_perm,xgb_gain,xgb_rank_gain,nn_perm,nn_rank_perm
11,remainder__capital-gain,0.022571,1.0,0.019074,5.0,-0.000061,274.0
2,cat__marital-status,0.016276,2.0,0.182600,2.0,0.001791,19.0
36,cat__marital-status cat__occupation,0.008373,3.0,0.009872,10.0,-0.000553,340.0
357,num__age num__educational-num num__hours-per-week,0.008169,4.0,0.037859,3.0,0.000614,130.0
50,cat__occupation num__educational-num,0.007268,5.0,0.013193,7.0,-0.000399,325.0
...,...,...,...,...,...,...,...
218,cat__marital-status cat__occupation remainder_...,-0.000450,373.0,0.003764,34.0,0.000717,111.0
57,cat__relationship num__age,-0.000481,374.0,0.000475,283.0,0.000317,188.0
19,cat__workclass num__age,-0.000532,375.0,0.000739,224.0,-0.000972,367.0
102,cat__workclass cat__marital-status cat__occupa...,-0.000645,376.0,0.000794,210.0,0.001771,20.0


# Get Final Feature Set

- poly models did better, so we want the best poly features

In [211]:
# select final features (INSERT_METHOD performed better)
rank_cols = [c for c in combined_poly.columns if "rank" in c]
combined_poly["mean_rank"] = combined_poly[rank_cols].mean(axis=1)
top_features = combined_poly.nsmallest(20, "mean_rank")["feature"].tolist()

['cat__marital-status',
 'cat__occupation cat__relationship remainder__gender',
 'cat__workclass cat__occupation',
 'cat__relationship cat__race num__age',
 'num__age num__educational-num num__hours-per-week',
 'cat__workclass cat__education num__educational-num',
 'cat__education cat__marital-status num__age',
 'cat__marital-status num__age',
 'cat__relationship num__age num__hours-per-week',
 'cat__occupation cat__race remainder__capital-gain',
 'cat__marital-status num__hours-per-week',
 'cat__workclass num__educational-num remainder__gender',
 'cat__marital-status remainder__capital-gain',
 'cat__occupation cat__native-country num__educational-num',
 'cat__occupation cat__race num__educational-num',
 'cat__education cat__occupation cat__relationship',
 'cat__workclass cat__relationship num__educational-num',
 'cat__education cat__native-country num__age',
 'num__educational-num num__hours-per-week remainder__gender',
 'cat__occupation remainder__gender']

In [219]:
X_train_final = X_train_poly[top_features]
X_train_final.head()

,cat__marital-status,cat__occupation cat__relationship remainder__gender,cat__workclass cat__occupation,cat__relationship cat__race num__age,num__age num__educational-num num__hours-per-week,cat__workclass cat__education num__educational-num,cat__education cat__marital-status num__age,cat__marital-status num__age,cat__relationship num__age num__hours-per-week,cat__occupation cat__race remainder__capital-gain,cat__marital-status num__hours-per-week,cat__workclass num__educational-num remainder__gender,cat__marital-status remainder__capital-gain,cat__occupation cat__native-country num__educational-num,cat__occupation cat__race num__educational-num,cat__education cat__occupation cat__relationship,cat__workclass cat__relationship num__educational-num,cat__education cat__native-country num__age,num__educational-num num__hours-per-week remainder__gender,cat__occupation remainder__gender
34342,0.330945,0.011034,0.029597,0.885603,0.0,0.000000,0.429899,1.323779,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.003583,0.000000,1.166953,0.0,0.042664
18559,0.330945,0.000000,0.078476,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.000101,0.000000,0.000000,0.0,0.000000
12477,0.456632,0.040719,0.070166,0.344644,0.0,0.000000,0.148292,0.456632,0.402580,0.0,0.456632,0.000000,0.0,0.000000,0.000000,0.013223,0.000000,0.021011,0.0,0.101144
560,0.031684,0.000000,0.079825,0.030034,0.0,0.000000,0.030869,0.095053,0.315717,0.0,0.031684,0.000000,0.0,0.000000,0.000000,0.003933,0.000000,0.875215,0.0,0.000000
3427,0.456632,0.050435,0.086909,0.344644,2.0,0.227899,0.075005,0.456632,0.402580,0.0,0.456632,1.387454,0.0,0.225086,0.214499,0.008284,0.558561,0.147559,2.0,0.125278


In [218]:
X_test_final = X_test_poly[top_features]
X_test_final.head()

,cat__marital-status,cat__occupation cat__relationship remainder__gender,cat__workclass cat__occupation,cat__relationship cat__race num__age,num__age num__educational-num num__hours-per-week,cat__workclass cat__education num__educational-num,cat__education cat__marital-status num__age,cat__marital-status num__age,cat__relationship num__age num__hours-per-week,cat__occupation cat__race remainder__capital-gain,cat__marital-status num__hours-per-week,cat__workclass num__educational-num remainder__gender,cat__marital-status remainder__capital-gain,cat__occupation cat__native-country num__educational-num,cat__occupation cat__race num__educational-num,cat__education cat__occupation cat__relationship,cat__workclass cat__relationship num__educational-num,cat__education cat__native-country num__age,num__educational-num num__hours-per-week remainder__gender,cat__occupation remainder__gender
40342,0.464428,0.000000,0.069188,0.019556,0.0,0.000000,0.588558,1.857713,0.197359,0.00000,0.464428,0.000000,0.000000,0.000000,0.000000,0.001554,0.000000,1.132620,0.0,0.000000
47680,0.326134,0.003283,0.006823,0.024474,0.0,0.000000,0.103325,0.326134,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.001040,0.000000,0.283155,0.0,0.114137
524,0.464428,0.049745,0.007288,1.388681,12.0,0.013744,0.427109,1.857713,4.896305,0.00000,1.393285,0.059781,0.000000,0.108962,0.103733,0.011437,0.024392,0.821928,3.0,0.121916
8508,0.464428,0.050330,0.085861,1.388681,0.0,0.000000,0.588558,1.857713,0.000000,329.23567,0.000000,0.000000,1456.911557,0.000000,0.000000,0.015945,0.000000,1.132620,0.0,0.123349
31692,0.464428,0.007978,0.001169,1.041511,6.0,0.003978,0.046353,1.393285,1.224076,0.00000,0.464428,0.119562,0.000000,0.034948,0.033271,0.000265,0.048784,0.089201,2.0,0.019552


# Tuning

In [214]:
def objective(trial):
    # Tree structure
    max_depth = trial.suggest_int("max_depth", 3, 12)
    min_child_weight = trial.suggest_float("min_child_weight", 1, 20, log=True)
    min_split_loss = trial.suggest_float("min_split_loss", 1e-8, 5.0, log=True)  # gamma

    # Learning rate & boosting rounds
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    n_estimators = trial.suggest_int("n_estimators", 100, 1000, step=50)

    # Sampling (helps a lot with redundant polynomial features)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.3, 1.0)
    colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.5, 1.0)

    # Regularization (important when polynomial features create multicollinearity)
    reg_alpha = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)   # L1
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True) # L2

    cls_optuna = XGBClassifier(
        random_state=SEED,
        enable_categorical=True,
        device="cuda:0",
        tree_method="hist",
        max_depth=max_depth,
        min_child_weight=min_child_weight,
        min_split_loss=min_split_loss,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        colsample_bylevel=colsample_bylevel,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
    )

    cv_scores = cross_val_score(
        cls_optuna, X_train_final, y_train,
        cv=skf, scoring='balanced_accuracy', n_jobs=1
    )
    return cv_scores.mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective, n_trials=50)

[I 2026-04-24 22:14:58,182] A new study created in memory with name: no-name-2f5ad89e-aaee-4934-962b-243ff3f9fee2
[I 2026-04-24 22:15:20,630] Trial 0 finished with value: 0.7720415002779795 and parameters: {'max_depth': 10, 'min_child_weight': 8.04857107857441, 'min_split_loss': 0.29783333893714814, 'learning_rate': 0.07167714089456216, 'n_estimators': 950, 'subsample': 0.5915149594625693, 'colsample_bytree': 0.3564584317454474, 'colsample_bylevel': 0.9653847722208282, 'reg_alpha': 4.201263442007305e-06, 'reg_lambda': 3.916911061826856e-07}. Best is trial 0 with value: 0.7720415002779795.
[I 2026-04-24 22:15:25,272] Trial 1 finished with value: 0.7716102889061515 and parameters: {'max_depth': 5, 'min_child_weight': 1.1926064246992512, 'min_split_loss': 2.4786774518753307, 'learning_rate': 0.023695298959423988, 'n_estimators': 500, 'subsample': 0.8497991349566757, 'colsample_bytree': 0.3364095967008806, 'colsample_bylevel': 0.6526747998109499, 'reg_alpha': 0.0012619834071349609, 'reg_la

In [217]:
def objective_mlp(trial):
    # Architecture
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layer_size = trial.suggest_int("layer_size", 32, 256, step=32)
    # Funnel architecture: each layer halves (common, effective default)
    hidden_layer_sizes = tuple(max(16, layer_size // (2**i)) for i in range(n_layers))

    # Activation
    activation = trial.suggest_categorical("activation", ["relu", "tanh"])

    # Optimizer & learning rate
    learning_rate = trial.suggest_categorical("learning_rate", ["adaptive", "invscaling", "constant"])
    solver = trial.suggest_categorical("solver", ["adam"])  # adam is almost always best for this
    learning_rate_init = trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)

    # Regularization (critical for polynomial features)
    alpha = trial.suggest_float("alpha", 1e-6, 1e-1, log=True)  # L2 penalty

    # Batch size
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])

    # Training control
    max_iter = trial.suggest_int("max_iter", 100, 500, step=50)

    # Early stopping (helps prevent overfitting on noisy poly features)
    early_stopping = True
    validation_fraction = 0.1
    n_iter_no_change = 15

    mlp_optuna = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate_init=learning_rate_init,
        alpha=alpha,
        learning_rate=learning_rate,
        batch_size=batch_size,
        max_iter=max_iter,
        early_stopping=early_stopping,
        validation_fraction=validation_fraction,
        n_iter_no_change=n_iter_no_change,
        random_state=SEED,
    )

    mlp_pipe = Pipeline(
        [
            ("scale", ColumnTransformer(transformers=[
               ("std", StandardScaler(), make_column_selector(dtype_include="number"))
            ])),
            ("model", mlp_optuna)
        ]
    )

    cv_scores = cross_val_score(
        mlp_pipe, X_train_final, y_train,
        cv=skf, scoring='balanced_accuracy', n_jobs=-1
    )
    return cv_scores.mean()

study_mlp = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5) # early stop bad trials
)
study_mlp.optimize(objective_mlp, n_trials=50)

print("Best params:", study_mlp.best_params)
print("Best score:", study_mlp.best_value)

[I 2026-04-24 22:29:31,649] A new study created in memory with name: no-name-40b2d0b9-8806-4760-8eb4-6d6f9efdcb88


ERROR! Session/line number was not unique in database. History logging moved to new session 881


[I 2026-04-24 22:29:52,335] Trial 0 finished with value: 0.7540180277483263 and parameters: {'n_layers': 3, 'layer_size': 224, 'activation': 'relu', 'learning_rate': 'invscaling', 'solver': 'adam', 'learning_rate_init': 0.0006737170693645698, 'alpha': 0.00020452112337117907, 'batch_size': 128, 'max_iter': 500}. Best is trial 0 with value: 0.7540180277483263.
[I 2026-04-24 22:30:03,709] Trial 1 finished with value: 0.7378360764426318 and parameters: {'n_layers': 1, 'layer_size': 128, 'activation': 'tanh', 'learning_rate': 'invscaling', 'solver': 'adam', 'learning_rate_init': 0.00011207098278563353, 'alpha': 0.0014865604732387685, 'batch_size': 256, 'max_iter': 200}. Best is trial 0 with value: 0.7540180277483263.
[I 2026-04-24 22:30:16,732] Trial 2 finished with value: 0.7485821461670439 and parameters: {'n_layers': 3, 'layer_size': 192, 'activation': 'relu', 'learning_rate': 'invscaling', 'solver': 'adam', 'learning_rate_init': 0.0021866128521646155, 'alpha': 0.0008652051425234911, 'ba

Best params: {'n_layers': 3, 'layer_size': 128, 'activation': 'relu', 'learning_rate': 'invscaling', 'solver': 'adam', 'learning_rate_init': 0.00018949514668972902, 'alpha': 0.00011778343247191801, 'batch_size': 256, 'max_iter': 400}
Best score: 0.7620603100598886


# Metamodel

In [223]:
# --- Reconstruct best params from Optuna studies ---
best_xgb_params = study_xgb.best_params
best_mlp_params = study_mlp.best_params

# Rebuild MLP architecture from the tuned n_layers + layer_size
n_layers = best_mlp_params.pop("n_layers")
layer_size = best_mlp_params.pop("layer_size")
hidden_layer_sizes = tuple(max(16, layer_size // (2**i)) for i in range(n_layers))

# --- Base models with tuned hyperparameters ---
def make_xgb():
    return XGBClassifier(
        random_state=SEED,
        enable_categorical=True,
        device="cuda:0",
        tree_method="hist",
        eval_metric="logloss",
        **best_xgb_params,
    )

def make_mlp():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layer_sizes,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=SEED,
            **best_mlp_params,
        )),
    ])

# Meta model
meta_model = LogisticRegression(max_iter=1000)

# OOF setup => `skf` declared earlier in (get training data)

# Store OOF predictions
xgb_oof = np.zeros(len(X_train_final))
mlp_oof = np.zeros(len(X_train_final))

# --- Generate OOF predictions ---
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_final, y_train), 1):
    X_tr, X_val = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]
    y_tr = y_train.iloc[train_idx]

    # Fresh model instances per fold (important — avoids leakage from prior fits)
    fold_xgb = make_xgb()
    fold_mlp = make_mlp()

    fold_xgb.fit(X_tr, y_tr)
    fold_mlp.fit(X_tr, y_tr)

    xgb_oof[val_idx] = fold_xgb.predict_proba(X_val)[:, 1]
    mlp_oof[val_idx] = fold_mlp.predict_proba(X_val)[:, 1]

# --- Train metamodel on OOF predictions ---
X_meta_train = np.column_stack((xgb_oof, mlp_oof))
meta_model.fit(X_meta_train, y_train)

# --- Retrain base models on FULL training data ---
final_xgb = make_xgb()
final_mlp = make_mlp()
final_xgb.fit(X_train_final, y_train)
final_mlp.fit(X_train_final, y_train)

# --- Generate test predictions ---
xgb_test = final_xgb.predict_proba(X_test_final)[:, 1]
mlp_test = final_mlp.predict_proba(X_test_final)[:, 1]

# --- Metamodel final predictions ---
X_meta_test = np.column_stack((xgb_test, mlp_test))
meta_preds = meta_model.predict(X_meta_test)

# --- Evaluate ---
print("\n=== Results ===")
print(f"XGB alone:      {balanced_accuracy_score(y_test, final_xgb.predict(X_test_final)):.4f}")
print(f"MLP alone:      {balanced_accuracy_score(y_test, final_mlp.predict(X_test_final)):.4f}")
print(f"Stacked model:  {balanced_accuracy_score(y_test, meta_preds):.4f}")

# Inspect what the metamodel learned
print(f"\nMeta model coefficients: XGB={meta_model.coef_[0][0]:.3f}, MLP={meta_model.coef_[0][1]:.3f}")
print(f"Meta model intercept: {meta_model.intercept_[0]:.3f}")


=== Results ===
XGB alone:      0.7261
MLP alone:      0.7754
Stacked model:  0.7293

Meta model coefficients: XGB=5.920, MLP=0.604
Meta model intercept: -3.261


In [ ]:
# comparing balanced accuracy scores across adult_fe data
print("NN Test Balanced Accuracy:", balanced_accuracy_score(y_test, final_mlp.predict(y_test)))
print("XGB Test Balanced Accuracy:", balanced_accuracy_score(y_test, final_xgb.predict(y_test)))
print("Stacked Model Test Balanced Accuracy:", balanced_accuracy_score(y_test, meta_preds))

# Discussion

**how performance changed as you reduced features**

- The FE models performed at 72 (XGB) and 76 (NN) balanced accuracy on the base model.
- For polynomial features, the NN increased very slightly. The XGB became much better. Thus we will use the poly set for our final model.
- For the final feature set with tuning, the XGB, again, improve significantly. The NN actually decreased in score by a decent amount. I suspect this is because neural networks actually *like* lots of features.

**which features were consistently important**

- `capital-gain`, `marital-status` we the top two in both sets.
- `education` was third most important in both sets, but in the polynomial set it was an interaction involving `education`.
- An interesting observation is that, in the poly set, the models had completely inverse preferences (in terms of ranking) but the top ranked features always involved the same few base features named above. The NN loves complex interactions while the XGB liked simpler features and interactions.

**which features you removed and why**

- The FE models performed decently well on 13 features...if performance gains were linearly related to the number of features, the polynomial set (377 features) should have resulted in perfect scores. This did not happen. Thus, we can assume that only a small subset of this features are high-signal. As we saw in the above answer, the top \approx 20 features were aligned. These will be our final feature set.

**whether stacking improved performance**

- Stacking significantly reduced performance.
- Strangely, the final NN train for the metamodel did way better than the tuning `cv_scores` led us to believe.

**and what you learned about how your model behaves**

From this activity, it seems like the tree-based model performed better when sufficiently tuned and given a sparser and more straightforward feature set, with some improvements when it we give it a head-start in finding interactions. On the other hand, the neural network had modest performance with simple features, but liked complex and inscrutable features when given the opportunity.




